# 04 — Buffer de eventos (Fase 0)

**Objetivo:** Reglas MVP sobre tracks → `events.jsonl` (simula cola Redis).


## Prerrequisitos

**03** → `outputs/03_track/tracks.jsonl`


## 1. Setup


In [1]:
from __future__ import annotations

import os
import uuid
from collections import defaultdict

from _common.io import (
    append_jsonl,
    bbox_centroid,
    default_roi,
    ensure_scripts_on_path,
    env_or_none,
    load_dotenv_repo,
    point_in_roi,
    read_json,
    read_jsonl,
    setup_logging,
    stage_output_dir,
    utc_now_iso,
)
from loguru import logger

ensure_scripts_on_path()
load_dotenv_repo()
setup_logging()


## 2. Configuration


In [2]:
TRACKS_PATH = stage_output_dir("03_track") / "tracks.jsonl"
CAPTURE_META = stage_output_dir("01_capture") / "metadata.json"
OUT_DIR = stage_output_dir("04_events")
EVENTS_PATH = OUT_DIR / "events.jsonl"

ROI = default_roi()
IDLE_SEC = 3.0
IDLE_PIX = 25.0
FPS_ASSUME = 10.0
VEHICLE_CLASSES = {"car", "truck", "forklift"}
REDIS_URL = env_or_none("REDIS_URL")
BACKEND = "redis" if REDIS_URL else "json"


## 3. Cargar tracks y dimensiones


In [3]:
rows = read_jsonl(TRACKS_PATH)
if not rows:
    raise RuntimeError("Sin tracks. Ejecute notebook 03.")

frame_w, frame_h = 1920, 1080
if CAPTURE_META.is_file():
    meta = read_json(CAPTURE_META)
    frame_w = int(meta.get("width", frame_w))
    frame_h = int(meta.get("height", frame_h))

by_track: dict[int, list] = defaultdict(list)
for r in rows:
    by_track[int(r["track_id"])].append(r)
logger.info("Tracks únicos: {}", len(by_track))


21:59:13 | INFO | Tracks únicos: 4


## 4. Motor de reglas MVP


In [4]:
if EVENTS_PATH.exists():
    EVENTS_PATH.unlink()

events_emitted = 0
idle_frames = int(IDLE_SEC * FPS_ASSUME)

for track_id, track_rows in by_track.items():
    track_rows.sort(key=lambda x: x["frame_idx"])
    cls = str(track_rows[0].get("class", "unknown"))

    # Regla: persona en ROI → warning
    if cls == "person":
        for r in track_rows:
            cx, cy = bbox_centroid(r["bbox"])
            if point_in_roi(cx, cy, ROI, frame_w, frame_h):
                append_jsonl(
                    EVENTS_PATH,
                    {
                        "event_id": str(uuid.uuid4()),
                        "type": "warning",
                        "severity": "medium",
                        "track_id": track_id,
                        "class": cls,
                        "zone": "roi_placeholder",
                        "frame_idx": r["frame_idx"],
                        "message": f"Persona {track_id} en zona ROI",
                        "ts": utc_now_iso(),
                    },
                )
                events_emitted += 1
                break

    # Regla: idle — poco movimiento durante N frames
    if len(track_rows) >= idle_frames:
        still = True
        ref = bbox_centroid(track_rows[0]["bbox"])
        for r in track_rows[1:idle_frames]:
            cx, cy = bbox_centroid(r["bbox"])
            if abs(cx - ref[0]) + abs(cy - ref[1]) > IDLE_PIX:
                still = False
                break
        if still:
            append_jsonl(
                EVENTS_PATH,
                {
                    "event_id": str(uuid.uuid4()),
                    "type": "idle",
                    "severity": "low",
                    "track_id": track_id,
                    "class": cls,
                    "zone": "floor",
                    "message": f"Track {track_id} inactivo ~{IDLE_SEC}s",
                    "ts": utc_now_iso(),
                },
            )
            events_emitted += 1

    # Regla: vehículo en movimiento → forklift_zone
    if cls in VEHICLE_CLASSES and len(track_rows) >= 2:
        c0 = bbox_centroid(track_rows[0]["bbox"])
        c1 = bbox_centroid(track_rows[-1]["bbox"])
        dist = abs(c1[0] - c0[0]) + abs(c1[1] - c0[1])
        if dist > IDLE_PIX:
            append_jsonl(
                EVENTS_PATH,
                {
                    "event_id": str(uuid.uuid4()),
                    "type": "forklift_zone",
                    "severity": "high",
                    "track_id": track_id,
                    "class": cls,
                    "zone": "aisle",
                    "message": f"Vehículo {track_id} en movimiento (proxy montacargas)",
                    "ts": utc_now_iso(),
                },
            )
            events_emitted += 1

logger.info("Eventos emitidos: {}", events_emitted)


21:59:13 | INFO | Eventos emitidos: 4


## 5. Backend opcional Redis


In [5]:
redis_pushed = 0
if BACKEND == "redis" and REDIS_URL:
    try:
        import redis
        r = redis.from_url(REDIS_URL)
        for ev in read_jsonl(EVENTS_PATH):
            r.rpush("visionops:events", __import__("json").dumps(ev))
            redis_pushed += 1
        logger.info("Redis: {} eventos en cola visionops:events", redis_pushed)
    except Exception as exc:
        logger.warning("Redis no disponible ({}); solo JSON local", exc)
else:
    logger.info("Backend JSON local: {}", EVENTS_PATH)


21:59:13 | INFO | Backend JSON local: /Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/outputs/04_events/events.jsonl


## 6. Validación


In [6]:
events = read_jsonl(EVENTS_PATH)
print(f"OK — {len(events)} eventos en {EVENTS_PATH}")
events[:3]


OK — 4 eventos en /Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/outputs/04_events/events.jsonl


[{'event_id': 'fc5bf44f-8ce2-48cd-bd1f-dea1c484fb56',
  'type': 'idle',
  'severity': 'low',
  'track_id': 1,
  'class': 'person',
  'zone': 'floor',
  'message': 'Track 1 inactivo ~3.0s',
  'ts': '2026-05-19T03:59:13.198125+00:00'},
 {'event_id': '176ad336-e8eb-458a-afed-32ed19b2ba66',
  'type': 'idle',
  'severity': 'low',
  'track_id': 2,
  'class': 'person',
  'zone': 'floor',
  'message': 'Track 2 inactivo ~3.0s',
  'ts': '2026-05-19T03:59:13.198423+00:00'},
 {'event_id': '4f3ce97e-9425-485a-ac6e-fd62ca06c0db',
  'type': 'warning',
  'severity': 'medium',
  'track_id': 3,
  'class': 'person',
  'zone': 'roi_placeholder',
  'frame_idx': 2,
  'message': 'Persona 3 en zona ROI',
  'ts': '2026-05-19T03:59:13.198497+00:00'}]

## Siguiente paso

Ramas paralelas: **[05_semantic_event.ipynb](05_semantic_event.ipynb)**, **[06_generate_heatmap.ipynb](06_generate_heatmap.ipynb)**, **[07_telegram_webhook.ipynb](07_telegram_webhook.ipynb)**
